# Datathon · стартовая точка

Сначала прочитайте START_HERE.md и brief. Этот notebook запускается на **учебных синтетических данных**, не на кейсе ForteBank. Результаты нельзя использовать как соревновательные. Kernel: Datathon (Python 3.12).

In [ ]:
from pathlib import Path
import os
import sys

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'pyproject.toml').exists(), 'Откройте notebook из папки datathon'
os.chdir(root)
print('Project:', root)
print('Python:', sys.executable)

## 1. Создать учебные данные

Для настоящего кейса замените путь на data/raw и сначала проверьте правила подготовленного кода, API и внешних данных.

In [ ]:
import subprocess
subprocess.run([sys.executable, 'scripts/make_synthetic.py'], check=True)

## 2. Описательный аудит

Полный аудит показывает схему и качество. Связи с target, сегменты и отбор признаков исследуйте только на train/development после разбиения.

In [ ]:
from datathon.eda import audit
from datathon.data import load_table

DATA = 'data/synthetic/train.csv'
report = audit(DATA, target='target', output='reports/eda/notebook')
frame = load_table(DATA)
frame.shape, report

## 3. Зафиксировать разбиение и доступные признаки

Time — прогноз будущего. Group — новые сущности. Random — независимые объекты. В шаблоне используется group; одинаковый клиент не попадёт по обе стороны.

In [ ]:
import yaml
from datathon.splits import make_split

CONFIG = 'configs/synthetic.yaml'
cfg = yaml.safe_load(Path(CONFIG).read_text())
train_idx, valid_idx, holdout_idx = make_split(frame, cfg)
print({name: len(rows) for name, rows in [('train', train_idx), ('validation', valid_idx), ('holdout', holdout_idx)]})
print('Features:', cfg['features'])

## 4. Один воспроизводимый прогон

Early stopping и порог выбираются на validation; итоговые метрики — на отдельном holdout. Не используйте этот holdout для цикла оптимизации.

In [ ]:
from datathon.train import train
run = train(CONFIG)
print(run)

In [ ]:
import json
import pandas as pd

metrics = json.loads((run / 'metrics.json').read_text())
pd.DataFrame({'Dummy': metrics['baseline_holdout'], 'CatBoost': metrics['holdout']})

## 5. Предсказания в порядке sample_submission

ID сверяются и выравниваются явно. Тип результата должен точно соответствовать формату организаторов.

In [ ]:
from datathon.predict import predict
predict(run, 'data/synthetic/test.csv', 'data/synthetic/sample_submission.csv', 'artifacts/practice_submission.csv', 'target', kind='probability')
pd.read_csv('artifacts/practice_submission.csv').head()

## 6. Перенос на настоящий кейс

Скопируйте configs/competition.example.yaml. Заполните цель, allowlist признаков и split по brief. Метрику, название положительного класса и цену ошибки проверьте отдельно. Если задача не табличная, измените постановку и baseline. Следующий материал: docs/AI_WORKFLOW.md и docs/REQUIREMENTS.md.